---
# Chapter 9 — What Did We Leave Unfinished?

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 9: What Did We Leave Unfinished? |
| Central question | What did the past leave unfinished that present work must still satisfy? |
| Main concepts | Open loop, Expected transition, Search footprint, Status maintenance |
| Implementation | open_loops |
| Experiment | ch9-20260920T000629Z-open-loops |
| Evidence status | Book result: conditional |
| Depends on | Chapter 8 (temporal state) |

---


## What this notebook demonstrates

Unfinished work becomes an expectation with a valid closing transition, not a mention count. The notebook:

1. **Freezes the canonical history** into an append-only event log
2. **Resolves expectation status** (`resolve_status`): SATISFIED, OPEN, CANCELLED — with closing evidence and tier
3. **Shows search footprints**: which sources were searched before concluding anything
4. **Contrasts mention counting against expected transitions** (`exp_e9a_mention`)
5. **Loads the frozen E9 metrics**

> **Evidence status**: Book result (conditional). Expected-transition resolution reaches canonical status accuracy 1.0 with false-closed rate 0.0 on the controlled fixture; the mention baseline trails at precision/recall 0.8.


## The chapter question

> **What did the past leave unfinished?**

Absence of evidence is not evidence of absence. A loop is open only when its expected closing transition has not been observed *and* the places it could have been recorded were actually searched.


## Concepts in this chapter


In [ ]:
import sys
from pathlib import Path


def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent


REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(9)
concepts = (meta.get("chapter", {}).get("concepts")
            or meta.get("concepts", []))
render_table([
    {"Concept ID": c["id"], "Name": c["name"], "Status": c["status"]}
    for c in concepts
], "Chapter 9 Concepts")

## The running example

The backup configuration still targets SQLite after the migration (`session-051`), fixture scans still encode old assumptions, docs obligations linger. Each is an expectation opened by one artifact and closable only by a specific later transition — or cancellable, or superseded.


## The mechanism: expectations over events

An `Expectation` names a desired end state plus the transitions that may close it. `resolve_status` replays the log up to a standpoint, collects candidate closings in tiers (observed state change, explicit authoritative markers, side effects), and returns a `StatusReport` with status, confidence class, opening/closing evidence, closure tier and a search footprint. Nothing is inferred from mere mention.


In [ ]:
from open_loops import fixtures as FX
from open_loops.status import resolve_status

events, texts = FX.canonical_history()
log = FX.freeze(events)
print(f"Canonical history: {len(log.events())} events, {len(texts)} text spans")

STANDPOINT = "2026-09-20T12:00:00Z"
for name in ("EXP_BACKUP", "EXP_DOCS", "EXP_FIXTURES", "EXP_STAGING"):
    exp = getattr(FX, name)()
    rep = resolve_status(exp, log, STANDPOINT, texts=texts)
    print(f"\n{name}: {rep.status} [{rep.confidence_class}]")
    print(f"  closure tier: {rep.closure_tier}")
    print(f"  closing evidence: {rep.closing_evidence}")
    print(f"  footprint: searched {rep.footprint.searched_sources}")

In [ ]:
# Mention counting vs expected transitions. The mention baseline flags
# any remark about a topic as open; the ledger truth disagrees twice,
# in opposite directions.
from open_loops.experiments import exp_e9a_mention

e9a = exp_e9a_mention()
print("Question:", e9a["question"])
print("Mention-open set:", e9a["M0_open"])
print("Transition-open set:", e9a["M1_open"])
print("Mention false positives:", e9a["M0_false_positives"])
print("Genuine recall (mention):", e9a["M0_recall_genuine"])

In [ ]:
from notebooks.memory._support import load_frozen_run

metrics = load_frozen_run("ch9-20260920T000629Z-open-loops")["metrics"]
render_table(
    [{"Metric": k, "Value": v} for k, v in metrics.items()],
    "Frozen E9 metrics (controlled fixture)")

## What happened?

Expected-transition resolution holds canonical status 1.0 with zero false closes while the mention baseline stalls at 0.8/0.8 with errors in both directions — a never-accepted remark flagged open, an issue-phrased obligation missed. The footprint is what makes 'no completion found' an auditable claim rather than an absence claim.


## Connect this to the experiment

The frozen run adds cross-artifact closure, side-effect completion, re-verification repair, bitemporal status and deadline flags — all 1.0 on the fixture — plus search-footprint coverage and projection-rebuild equivalence. The dissenting unit the corpus keeps (`mb-refutes-ch9-scope`) bounds the claim: fixture-level accuracy with oracle-labelled expectations, not extraction from raw history.


## What this establishes

- **Open loops are expectations with closings**, not mentions
- **Footprints separate absence of evidence from evidence of absence**
- **Status derivation and status maintenance agree on quality** here; maintenance buys cheaper listing at staleness risk
- **Conditional**: the verdict is fixture-level


## What this does NOT establish

- Extraction of expectations from raw, unlabelled history
- Real-corpus open-loop quality
- Deadline and priority semantics beyond flags


In [ ]:
# TRY IT YOURSELF: move the standpoint earlier, before the closing
# transition exists, and watch the same expectation report OPEN.
rep_early = resolve_status(FX.EXP_BACKUP(), log, "2024-08-01T00:00:00Z",
                           texts=texts)
print(f"EXP_BACKUP at 2024-08-01: {rep_early.status} "
      f"[{rep_early.confidence_class}]")
print(f"  closing evidence: {rep_early.closing_evidence}")

## Where this leads next

Chapter 10 asks what past matters right now: frames condition which expectations surface for current work.

> **See this chapter in code:** [Open the companion Jupyter notebook](memory 9-chapter.ipynb)
